# Coursework 1

This notebook is intended to be used as a starting point for your experiments. The instructions can be found in the MLP2024_25_CW1_Spec.pdf (see Learn,  Assignment Submission, Coursework 1). The methods provided here are just helper functions. If you want more complex graphs such as side by side comparisons of different experiments you should learn more about matplotlib and implement them. Before each experiment remember to re-initialize neural network weights and reset the data providers so you get a properly initialized experiment. For each experiment try to keep most hyperparameters the same except the one under investigation so you can understand what the effects of each are.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')

def train_model_and_plot_stats(
        model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True):
    
    # As well as monitoring the error over training also monitor classification
    # accuracy i.e. proportion of most-probable predicted classes being equal to targets
    data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

    # Use the created objects to initialise a new Optimiser instance.
    optimiser = Optimiser(
        model, error, learning_rule, train_data, valid_data, data_monitors, notebook=notebook)

    # Run the optimiser for num_epochs epochs (full passes through the training set)
    # printing statistics every epoch.
    stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)

    # Plot the change in the validation and training set error over training.
    fig_1 = plt.figure(figsize=(8, 4))
    ax_1 = fig_1.add_subplot(111)
    
    for k in ['error(train)', 'error(valid)']:
        ax_1.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]], label=k)
    ax_1.legend(loc=0)
    ax_1.set_xlabel('Epoch number')
    ax_1.set_ylabel('Error')

    # Plot the change in the validation and training set accuracy over training.
    fig_2 = plt.figure(figsize=(8, 4))
    ax_2 = fig_2.add_subplot(111)
    for k in ['acc(train)', 'acc(valid)']:
        ax_2.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]], label=k)
    ax_2.legend(loc=0)
    ax_2.set_xlabel('Epoch number')
    ax_2.set_xlabel('Accuracy')
    
    return stats, keys, run_time, fig_1, ax_1, fig_2, ax_2

In [ ]:
#This codes defined a custom training and plot function. Instead of making and showing a plot, it simply appends the data to an existing plot.

def train_model_and_plot_stats_custom(
        model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True, identifier = ''):
    
    # As well as monitoring the error over training also monitor classification
    # accuracy i.e. proportion of most-probable predicted classes being equal to targets
    data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

    # Use the created objects to initialise a new Optimiser instance.
    optimiser = Optimiser(
        model, error, learning_rule, train_data, valid_data, data_monitors, notebook=notebook)

    # Run the optimiser for num_epochs epochs (full passes through the training set)
    # printing statistics every epoch.
    stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)
    ptype = ['-' ,'--']
    # Plot the change in the validation and training set error over training.
    j = 0
    for k in ['error(train)'.format(identifier), 'error(valid)'.format(identifier)]:
        ax_1.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]],ptype[j], label=str(identifier) + k[5:])
        j+= 1
    

    # Plot the change in the validation and training set accuracy over training.
    j = 0
    for k in ['acc(train)'.format(identifier), 'acc(valid)'.format(identifier)]:
        ax_2.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]],ptype[j], label=str(identifier) + k[3:])
        j+= 1
    ax_2.legend(loc=0)
    ax_2.set_xlabel('Epoch number')
    ax_2.set_xlabel('Accuracy')
    
    return stats, keys, run_time, fig_1, ax_1, fig_2, ax_2

In [ ]:
#This function trains the model and returns only the stats.

def train_model_and_provide_stats(
        model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True):
    
    # As well as monitoring the error over training also monitor classification
    # accuracy i.e. proportion of most-probable predicted classes being equal to targets
    data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

    # Use the created objects to initialise a new Optimiser instance.
    optimiser = Optimiser(
        model, error, learning_rule, train_data, valid_data, data_monitors, notebook=notebook)

    # Run the optimiser for num_epochs epochs (full passes through the training set)
    # printing statistics every epoch.
    stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)

    # Plot the change in the validation and training set error over training.
    
    return stats

In [ ]:
# The below code will set up the data providers, random number
# generator and logger objects needed for training runs. As
# loading the data from file take a little while you generally
# will probably not want to reload the data providers on
# every training run. If you wish to reset their state you
# should instead use the .reset() method of the data providers.
import numpy as np
import logging
import sys
# sys.path.append('/path/to/mlpractical')
from mlp.data_providers import MNISTDataProvider, EMNISTDataProvider

# Seed a random number generator
seed = 11102019 
rng = np.random.RandomState(seed)
batch_size = 100
# Set up a logger object to print info about the training run to stdout
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

smooth_labels_indicator = None #change to True for labe smoothing

# Create data provider objects for the MNIST data set
train_data = EMNISTDataProvider('train', batch_size=batch_size, rng=rng)
valid_data = EMNISTDataProvider('valid', batch_size=batch_size, rng=rng, smooth_labels = smooth_labels_indicator)
test_data = EMNISTDataProvider('test', batch_size=batch_size, rng=rng) #This loads the test data and is not nescsarry for all but the last cell to run

In [ ]:
# The model set up code below is provided as a starting point.
# You will probably want to add further code cells for the
# different experiments you run.

%pip install tqdm

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser

# Setup hyperparameters
learning_rate = 0.001
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 32

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

# Create model with ONE hidden layer
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

# Remember to use notebook=False when you write a script to be run in a terminal
_ = train_model_and_plot_stats(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)

In [ ]:
#Code that plots the results for the model width investigation
#to plot show the plots, run > plt.show() in a cell below. This ensures the plot is visable and not hidden under the network stats

%pip install tqdm

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser

# Setup hyperparameters
learning_rate = 0.0009
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 32

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

# Create model with ONE hidden layer
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

fig_1 = plt.figure(figsize=(8, 4))
ax_1 = fig_1.add_subplot(111)

fig_2 = plt.figure(figsize=(8, 4))
ax_2 = fig_2.add_subplot(111)

# Remember to use notebook=False when you write a script to be run in a terminal
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'width 32')

hidden_dim = 64
weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'width 64')


hidden_dim = 128
weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # hidden layer
   
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'width 128')

ax_1.legend(loc=0)
ax_1.set_xlabel('Epoch number')
ax_1.set_ylabel('Error')

ax_2.legend(loc=0)
ax_2.set_xlabel('Epoch number')
ax_2.set_ylabel('Accuracy')

In [ ]:
plt.show()

In [ ]:
#Code that plots the results for the model depth investigation
#to plot show the plots, run > plt.show() in a cell below. This ensures the plot is visable and not hidden under the network stats

%pip install tqdm

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser

# Setup hyperparameters
learning_rate = 0.0009
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 32

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

# Create model with ONE hidden layer
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

fig_1 = plt.figure(figsize=(8, 4))
ax_1 = fig_1.add_subplot(111)

fig_2 = plt.figure(figsize=(8, 4))
ax_2 = fig_2.add_subplot(111)

# Remember to use notebook=False when you write a script to be run in a terminal
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'depth 1')

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # hidden layer 1
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # hidden layer 2
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'depth 2')


weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init), # first hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # second hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # third hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'depth 3')

ax_1.legend(loc=0)
ax_1.set_xlabel('Epoch number')
ax_1.set_ylabel('Error')

ax_2.legend(loc=0)
ax_2.set_xlabel('Epoch number')
ax_2.set_ylabel('Accuracy')

In [ ]:
plt.show()


In [ ]:
# Create model with TWO hidden layers
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init), # first hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # second hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

# Remember to use notebook=False when you write a script to be run in a terminal
stats = train_model_and_plot_stats(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)

In [ ]:
#code for dropout layer model
#to plot show the plots, run > plt.show() in a cell below. This ensures the plot is visable and not hidden under the network stats

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser
from mlp.penalties import L1Penalty, L2Penalty

# Setup hyperparameters
learning_rate = 1e-4
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

penalty = None

# Create model with ONE hidden layer
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init), # first hidden layer
    DropoutLayer(incl_prob = 0.7,rng = rng),
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # second hidden layer
    DropoutLayer(incl_prob = 0.7,rng = rng),
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # third hidden layer
    DropoutLayer(incl_prob = 0.7,rng = rng),
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

fig_1 = plt.figure(figsize=(8, 4))
ax_1 = fig_1.add_subplot(111)

fig_2 = plt.figure(figsize=(8, 4))
ax_2 = fig_2.add_subplot(111)

# Remember to use notebook=False when you write a script to be run in a terminal
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'L1Regularization')


In [ ]:
plt.show()

In [ ]:
#code for L1 and L2 models
#to plot show the plots, run > plt.show() in a cell below. This ensures the plot is visable and not hidden under the network stats

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser
from mlp.penalties import L1Penalty, L2Penalty

# Setup hyperparameters
learning_rate = 1e-4
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

penalty = L1Penalty(1e-3) #switch to L2Penalty(1e-3) for second regulatization investigation

# Create model with penalties
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, weights_penalty = penalty), # first hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty = penalty), # second hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty = penalty), # third hidden layer
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init, weights_penalty = penalty) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

fig_1 = plt.figure(figsize=(8, 4))
ax_1 = fig_1.add_subplot(111)

fig_2 = plt.figure(figsize=(8, 4))
ax_2 = fig_2.add_subplot(111)

# Remember to use notebook=False when you write a script to be run in a terminal
_ = train_model_and_plot_stats_custom(
    model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval,fig_1,ax_1,fig_2,ax_2, notebook=True,identifier = 'L1Regularization')


In [ ]:
plt.show()

In [ ]:
#code for L1 & L2 investigation. It trains the 2 models for different hyperparameter values and plots their performances.
#to plot show the plots, run > plt.show() in a cell below. This ensures the plot is visable and not hidden under the network stats

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser
from mlp.penalties import L1Penalty, L2Penalty

# Setup hyperparameters
learning_rate = 1e-4
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

l1_val_acc = []
l2_val_acc = []

l1_gap = []
l2_gap = []
penalty_coeffs = []
# Remember to use notebook=False when you write a script to be run in a terminal

for j in range(-5,0,1):

    penalty_coeffs.append(10**j)
    penalty = L1Penalty(10**j)
    # Create model with penalties
    model = MultipleLayerModel([
        AffineLayer(input_dim, hidden_dim, weights_init, weights_penalty = penalty, biases_penalty = penalty), # first hidden layer
        ReluLayer(),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty = penalty, biases_penalty = penalty), # second hidden layer
        ReluLayer(),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty = penalty, biases_penalty = penalty), # third hidden layer
        ReluLayer(),
        AffineLayer(hidden_dim, output_dim, weights_init, biases_init, weights_penalty = penalty, biases_penalty = penalty) # output layer
    ])
    
    l1stats = train_model_and_provide_stats(
            model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)
    
    l1_val_acc.append(l1stats[-1][3])
    l1_gap.append(l1stats[-1][2] - l1stats[-1][0])

    penalty = L2Penalty(10**j)
    # Create model with penalties
    model = MultipleLayerModel([
        AffineLayer(input_dim, hidden_dim, weights_init, weights_penalty = penalty, biases_penalty = penalty), # first hidden layer
        ReluLayer(),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty = penalty, biases_penalty = penalty), # second hidden layer
        ReluLayer(),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty = penalty, biases_penalty = penalty), # third hidden layer
        ReluLayer(),
        AffineLayer(hidden_dim, output_dim, weights_init, biases_init, weights_penalty = penalty, biases_penalty = penalty) # output layer
    ])
    
    l2stats = train_model_and_provide_stats(
            model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)
    
    l2_val_acc.append(l2stats[-1][3])
    l2_gap.append(l2stats[-1][2] - l2stats[-1][0])



plt.plot(penalty_coeffs,l1_val_acc,'-',label = 'L1 Val. Acc')
plt.plot(penalty_coeffs,l2_val_acc,'-',label = 'L2 Val. Acc')
plt.plot(penalty_coeffs,l1_gap,'--',label = 'L1 Gap')
plt.plot(penalty_coeffs,l2_gap,'--',label = 'L2 Gap')
plt.legend(loc=0)
plt.xlabel('Weight Decay Value')
plt.ylabel('Accuracy / Error value')
plt.xscale('log')


In [ ]:
plt.show()

In [ ]:
#code for dropout investigation. It trains the model for different hyperparameter values and plots the performances and relevent statistics.
#to plot show the plots, run > plt.show() in a cell below. This ensures the plot is visable and not hidden under the network stats

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser
from mlp.penalties import L1Penalty, L2Penalty

# Setup hyperparameters
learning_rate = 1e-4
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

val_acc = []
gap = []

p_values = []
# Remember to use notebook=False when you write a script to be run in a terminal

for j in range(1,11,1):

    p = j/10
    p_values.append(p)
    # Create model with penalties
    model = MultipleLayerModel([
        AffineLayer(input_dim, hidden_dim, weights_init), # first hidden layer
        DropoutLayer(incl_prob = p,rng = rng),
        ReluLayer(),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # second hidden layer
        DropoutLayer(incl_prob = p,rng = rng),
        ReluLayer(),
        AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # third hidden layer
        DropoutLayer(incl_prob = p,rng = rng),
        ReluLayer(),
        AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
    ])
    
    stats = train_model_and_provide_stats(
            model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)
    
    val_acc.append(stats[-1][3])
    gap.append(stats[-1][2] - stats[-1][0])
    
    



plt.plot(p_values,val_acc,'-',label = 'Model Accuracy')
plt.plot(p_values,gap,'--',label = 'Train Error Gap')
plt.legend(loc=0)
plt.xlabel('Dropout Parameter Value')
plt.ylabel('Accuracy / Error value')



In [ ]:
plt.show()

In [ ]:
#Program that trains the best performing model (dropout) on train data and evaluates it sperformance on the test data.

from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser
from mlp.penalties import L1Penalty, L2Penalty

# Setup hyperparameters
learning_rate = 1e-4
num_epochs = 100
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

penalty = None

# Create model with ONE hidden layer
model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init), # first hidden layer
    DropoutLayer(incl_prob = 0.97,rng = rng),
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # second hidden layer
    DropoutLayer(incl_prob = 0.97,rng = rng),
    ReluLayer(),
    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init), # third hidden layer
    DropoutLayer(incl_prob = 0.97,rng = rng),
    ReluLayer(),
    AffineLayer(hidden_dim, output_dim, weights_init, biases_init) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

optimiser = Optimiser(
        model, error, learning_rule, train_data, valid_data, data_monitors, notebook=True)

stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)
   
#test_vals = optimiser.eval_monitors(test_data.inputs,test_data.to_one_of_k(test_data.targets))

net_output = optimiser.model.fprop(test_data.inputs, evaluation = True)[10]
k_hot_test_labels = test_data.to_one_of_k(test_data.targets)

print('Test Error: {0}'.format(error(net_output,k_hot_test_labels)))

N = k_hot_test_labels.shape[0]

labels = test_data.targets

sm = SoftmaxLayer()
estimates = sm.fprop(net_output)
label_guesses = np.argmax(estimates,axis = 1)
accuracy = np.sum(1*(labels == label_guesses)) * 1/N

print('Accuracy = {0}'.format(accuracy))

